# Atmospheric CO₂ at Mauna Loa

The concentration of carbon dioxide in the atmosphere is the benchmark indicator of human-driven
climate change. This notebook presents the continuous record of **NOAA's Mauna Loa Observatory**
(Hawaiʻi, 3400 m), the longest direct record of atmospheric CO₂ (since 1958): the monthly
concentration, its annual growth rate and a summary table of the record.

**Data**
- NOAA Global Monitoring Laboratory, Mauna Loa: monthly means, annual means and annual growth rates
  ([gml.noaa.gov/ccgg/trends](https://gml.noaa.gov/ccgg/trends/)). Dr. Xin Lan, NOAA/GML and
  Dr. Ralph Keeling, Scripps Institution of Oceanography.

The projected CO₂ concentration of the SSP scenarios is shown in
[Projections › Greenhouse gases](../../../projections/Regional/greenhouse_gases/co2_ssps.ipynb).

In [1]:
# CIndRA notebook display standard
import matplotlib as mpl

mpl.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 15,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 13,
    'figure.dpi': 100,
    'savefig.dpi': 150,
})

In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import display

root_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next((p for p in root_candidates if (p / 'functions' / 'co2.py').is_file()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate the CIndRA repository root.')
sys.path.insert(0, str(repo_root / 'functions'))

from co2 import co2_summary_table, decadal_growth, download_noaa_co2, load_noaa_co2, plot_growth_rate, plot_keeling_curve
from summary_tables import save_summary_table

## 1. Configuration and data

Files are cached under `data/ghg/`; set `force_download = True` to refresh them (NOAA updates the
record every month).

In [3]:
force_download = False

data_dir = repo_root / 'data' / 'ghg'
fig_dir = repo_root / 'outputs' / 'figures' / 'global'
table_dir = repo_root / 'outputs' / 'tables' / 'global'
fig_dir.mkdir(parents=True, exist_ok=True)

noaa = load_noaa_co2(download_noaa_co2(data_dir, force=force_download))
print(f"Mauna Loa monthly record: {noaa['monthly'].time.min():%b %Y} – {noaa['monthly'].time.max():%b %Y}")
noaa['annual'].tail()

Mauna Loa monthly record: Mar 1958 – Sep 2026


,year,mean,unc
62,2021,416.41,0.12
63,2022,418.53,0.12
64,2023,421.08,0.12
65,2024,424.61,0.12
66,2025,427.35,0.12


## 2. Monthly concentration

The red line shows the monthly means, with the seasonal cycle caused by the uptake and release of
CO₂ by Northern Hemisphere vegetation; the black line removes that cycle.

In [4]:
fig, ax = plot_keeling_curve(noaa)
fig.savefig(fig_dir / 'GHG_co2_mauna_loa_monthly.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/88/z9xxhn052qddjc_ppp2j_8gr0000gn/T/ipykernel_33091/1830358785.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Growth rate

Annual increase of the CO₂ concentration (ppm per year) and its mean in each decade. Year-to-year
fluctuations are largely driven by El Niño (faster growth) and La Niña through the response of the
land biosphere.

In [5]:
fig, ax = plot_growth_rate(noaa)
fig.savefig(fig_dir / 'GHG_co2_mauna_loa_growth_rate.png', dpi=150, bbox_inches='tight')
plt.show()

growth_by_decade = decadal_growth(noaa['growth'])
save_summary_table(growth_by_decade.round(2), table_dir, 'GHG_co2_growth_rate_by_decade.csv')
growth_by_decade.round(2)

Saved summary table (8 rows) to /Users/laurac/Library/Mobile Documents/com~apple~CloudDocs/Projects/CC_indicators/PICCM/CIndRA/outputs/tables/global/GHG_co2_growth_rate_by_decade.csv


/var/folders/88/z9xxhn052qddjc_ppp2j_8gr0000gn/T/ipykernel_33091/1971542474.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Decade,Mean growth rate (ppm/yr),Years
0,1950s (1 yr),0.94,1
1,1960s,0.86,10
2,1970s,1.28,10
3,1980s,1.61,10
4,1990s,1.51,10
5,2000s,1.97,10
6,2010s,2.42,10
7,2020s (6 yr),2.56,6


## 4. Summary table

In [6]:
summary = co2_summary_table(noaa)
save_summary_table(summary, table_dir, 'GHG_co2_summary_table.csv')
noaa['annual'].to_csv(table_dir / 'GHG_co2_mauna_loa_annual.csv', index=False)
summary

Saved summary table (19 rows) to /Users/laurac/Library/Mobile Documents/com~apple~CloudDocs/Projects/CC_indicators/PICCM/CIndRA/outputs/tables/global/GHG_co2_summary_table.csv


,Metric,Value
0,ATMOSPHERIC CO2 – NOAA Mauna Loa Observatory,
1,"Average CO2 concentration, 1959–2025 (ppm)",361.251
2,Change in CO2 concentration since 1959 (ppm),110.35
3,Rate of change in CO2 concentration (ppm/decade),16.72
4,Trend p-value,< 0.001
5,Trend significant at 95%,Yes
6,Average CO2 concentration: 1959–1968 (ppm),319.423
7,Average CO2 concentration: 2016–2025 (ppm),415.373
8,Highest annual CO2 concentration (2025) (ppm),427.35
9,Lowest annual CO2 concentration (1959) (ppm),315.98


## Interpretation and reproducibility notes

- Mauna Loa measures well-mixed background air and is representative of the Northern Hemisphere;
  the global mean is ~1–2 ppm lower and the Southern Hemisphere (e.g. Cape Grim) a few ppm lower.
- Missing monthly values in the NOAA files (-99.99) are treated as gaps; NOAA revises the most recent
  months as calibrations are finalised.
- Trends in the summary table are ordinary least-squares fits of the annual means, a description of
  the record rather than of its accelerating growth (see the growth-rate figure).